In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json ,to_json,col,struct
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType

In [2]:
spark = SparkSession.builder \
 .appName("FraudDetection") \
 .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3") \
 .getOrCreate()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-175dde17-4b86-4fff-b547-2151f748e00a;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 1309ms :: artifacts dl 30ms
	:: modules in u

In [3]:
spark.sparkContext.setLogLevel("WARN")

In [4]:

user_df = spark.read.csv("data/user_table.csv", header=True ,inferSchema= True)


26/06/19 06:13:56 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [5]:
tx_schema = StructType([
     StructField("tx_id", IntegerType(), True),
     StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),

     StructField("timestamp", DoubleType(), True)

] )

In [6]:
kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()

In [7]:
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"),
tx_schema).alias("tx")).select("tx.*")

fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

In [8]:
fraud_stream1 = parsed_stream.filter(col("amount") > 5000.0)

In [9]:
enriched_fraud = fraud_stream.join(user_df, "userId")

In [10]:
enriched_fraud1 = fraud_stream1.join(user_df, "userId")

In [11]:
output_stream1 = enriched_fraud1 \
     .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")

In [12]:
output_stream = enriched_fraud \
     .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")

In [ ]:
query1 = output_stream1.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notification1") \
 .option("checkpointLocation", "/workspace/checkpoints/query1") \
 .start()
query = output_stream.writeStream \
 .format("kafka") \
 .option("kafka.bootstrap.servers", "kafka:9092") \
 .option("topic", "fraud-notification") \
 .option("checkpointLocation", "/workspace/checkpoints/query") \
 .start()
query.awaitTermination()
query1.awaitTermination()

26/06/19 06:14:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:14:05 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/06/19 06:14:06 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/06/19 06:14:06 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
                                                                                